In [78]:
!pip install pyserial

In [341]:
import serial, time
#!pip install pyserial

In [342]:
ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [343]:
print(serial)

<module 'serial' from 'C:\\Users\\caleb\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [344]:
print(serial.__file__)

C:\Users\caleb\anaconda3\Lib\site-packages\serial\__init__.py


In [345]:
print(serial.__version__)

3.5


In [346]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [347]:
baudrate = 115200

In [348]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [349]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [350]:
ser.in_waiting

0

In [351]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [352]:
read_all(ser)

''

In [353]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [354]:
read_one_line(ser)

''

In [355]:
read_all(ser)

''

In [356]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [357]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

# Break an integer into two bytes

In [358]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

In [359]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

In [360]:
def GrabberAngle(L1, L2, X, Z):
    Theta_intermediate1 = 90 - rtd * np.arctan2(Z, X)
    
    Theta_intermediate3 = rtd*np.arccos(((X**2 + Z**2) + L2**2 - L1**2)/(2*(X**2 + Z**2)**.5 * L2 ))
    Angle_Between = Theta_intermediate3 + Theta_intermediate1
    return Angle_Between

In [361]:
GrabberAngle(25, 25, 38, 20)


np.float64(93.05527299623972)

In [362]:
def XZLocation(l1, l2, X, Z):
    #Define Link lengths

    
    #distance from origin to tip
    r_squared = X**2 + Z**2
    
    #law of cos for angle between links
    alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
    
    alpha = np.arccos(alpha_temp)
    
    
    
    #vertical angle theorem for theta 2
    theta2 = 180 - alpha*rtd
    
    #triangle in link1 co-ordinant system for psi
    psi = np.arctan2(l2*sind(theta2),l1+l2*cosd(theta2))*rtd
    
    #angle of r to x-axis
    beta = np.arctan2(Z,X)*rtd
    
    #difference in beta and psi is theta 1
    theta1 = beta - psi
    if theta1 < 0:
        theta1 = theta1 +90
        theta2 = theta2 * -1
        
            
        
    return theta1, theta2
    

In [363]:
T1, T2 = XZLocation(27, 27, 38.18, 20)
print('\n',T1)
print('\n',T2)


 80.60281464980845

 -74.08856440695598


In [364]:
def findz(L1, L2, X):
    R_Square = L1**2 + L2**2
    Z = (R_Square -X**2)**.5
    return Z
    

In [365]:
def theta2interpolation(theta2): 
    #convert angle into arduino code
    theta_min = 0     # minimum angle
    theta_max = 180   # maximum angle
    min_new = 1000    # minimum servo value
    max_new = 2000    # maximum servo value

    #linear interpolate for the first theta value
    myint = min_new + ((theta2-theta_min)*(max_new-min_new))/(theta_max-theta_min)
    return myint

In [366]:
def theta3interpolation(theta3):
    #convert angle into arduino code
    theta_min = 0     # minimum angle
    theta_max = 180   # maximum angle
    min_new = 1000    # minimum servo value
    max_new = 2000    # maximum servo value
    #linear interpolate for the second theta value; 180 and 90 included to account for robots home position as The servo has 1000 -> theta2=90, 2000 -> theta2 = -90
    myint2 = min_new + (((180-(90+theta3))-theta_min)*(max_new-min_new))/(theta_max-theta_min)
    #print('\n',myint2)
    return myint2

In [367]:
def theta4interpolation(theta4):
    #convert angle into arduino code
    theta_min = 0     # minimum angle
    theta_max = 180   # maximum angle
    min_new = 2100    # minimum servo value
    max_new = 900    # maximum servo value
    #linear interpolate for the second theta value; 180 and 90 included to account for robots home position as The servo has 1000 -> theta2=90, 2000 -> theta2 = -90
    myint2 = min_new + (((180-(90+theta4))-theta_min)*(max_new-min_new))/(theta_max-theta_min)
    #print('\n',myint2)
    return myint2

In [395]:
L1 = 25
L2 = 25

ObjectPickupX = 30
ObjectPickupY = 28

ObstacleX = 5
ObstacleY = 8

DropOffX = -38
DropOffY = 38

Height1 = 20 # Hover Height
Height2 = 22 #Grab Height
CL = 2 #Obstacle Clearance

Theta1 = np.zeros(8)
Theta2 = np.zeros(8)
Theta3 = np.zeros(8)
Theta4 = np.zeros(8)

X = np.zeros(8)
Y = np.zeros(8)
Z = np.zeros(8)

Servo1 = np.zeros(8)
Servo2 = np.zeros(8)
Servo3 = np.zeros(8)
Servo4 = np.zeros(8)
Servo5 = np.zeros(8)

byte1 = np.zeros(8)
byte2 = np.zeros(8)
byte3 = np.zeros(8)
byte4 = np.zeros(8)
byte5 = np.zeros(8)
byte6 = np.zeros(8)
byte7 = np.zeros(8)
byte8 = np.zeros(8)
byte9 = np.zeros(8)
byte10 = np.zeros(8)


In [396]:
#Step 0 - Origin
Theta1[0] = 0
Theta2[0] = 90
Theta3[0] = 0
Theta4[0] =0

Servo1[0] = theta2interpolation(Theta1[0])
Servo2[0] = theta2interpolation(Theta2[0])
Servo3[0] = theta3interpolation(Theta3[0])
Servo4[0] = theta4interpolation(Theta4[0])


print('\n',Servo1[0])
print('\n',Servo2[0])
print('\n',Servo3[0])
print('\n',Servo4[0])
print('\n',Servo5[0])



 1000.0

 1500.0

 1500.0

 1500.0

 0.0


In [397]:
byte1[0], byte2[0] = break_into_two(Servo1[0])
byte3[0], byte4[0] = break_into_two(Servo2[0])
byte5[0], byte6[0] = break_into_two(Servo3[0])
byte7[0], byte8[0] = break_into_two(Servo4[0])
byte9[0], byte10[0] = break_into_two(Servo5[0])

In [398]:
WriteByte(ser, int(byte1[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[0]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[0]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[0]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[0]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")

 servo1=My int: 1000  
 servo1=My int2: 1500  
 servo1=My int3: 1500  
 servo1=My int4: 1500  
 servo1=My int5: 0  


In [399]:
#Step 1 - Go to correct angle to pick up object
Theta1[1] = rtd*np.arctan2(ObjectPickupY, ObjectPickupX)
Theta2[1] = Theta2[0]
Theta3[1] = Theta3[0]
Theta4[1] = Theta4[0]

Servo1[1] = theta2interpolation(Theta1[1])
Servo2[1] = theta2interpolation(Theta2[1])
Servo3[1] = theta3interpolation(Theta3[1])
Servo4[1] = theta4interpolation(Theta4[1])

print('\n',Servo1[1])
print('\n',Servo2[1])
print('\n',Servo3[1])
print('\n',Servo4[1])
print('\n',Servo5[1])



 1239.028144383989

 1500.0

 1500.0

 1500.0

 0.0


In [400]:
byte1[1], byte2[1] = break_into_two(Servo1[1])
byte3[1], byte4[1] = break_into_two(Servo2[1])
byte5[1], byte6[1] = break_into_two(Servo3[1])
byte7[1], byte8[1] = break_into_two(Servo4[1])
byte9[1], byte10[1] = break_into_two(Servo5[1])

In [401]:
WriteByte(ser, int(byte1[1]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[1]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[1]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[1]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[1]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[1]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[1]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[1]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[1]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[1]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")

 servo1=My int: 1239  
 servo1=My int2: 1500  
 servo1=My int3: 1500  
 servo1=My int4: 1500  
 servo1=My int5: 0  


In [402]:
#Step 2 - Go to correct height and radius to pick up object
Theta1[2] = Theta1[1]
X[2] = (ObjectPickupY**2 + ObjectPickupX**2)**.5
Z[2] = Height1
Theta2[2], Theta3[2] = XZLocation(L1, L2, X[2], Z[2])
Theta4[2] = -1 * (180 - GrabberAngle(L1, L2, X[2], Z[2]))
ThetaTest = GrabberAngle(L1, L2, X[2], Z[2])
Servo1[2] = theta2interpolation(Theta1[2])
Servo2[2] = theta2interpolation(Theta2[2])
Servo3[2] = theta3interpolation(Theta3[2])
Servo4[2] = theta4interpolation(Theta4[2])

'''
print('\n',Servo1[2])
print('\n',Servo2[2])
print('\n',Servo3[2])
print('\n',Servo4[2])
print('\n',Servo5[2])
'''
print('\n',ThetaTest)
print('\n',Theta2[2])
print('\n',Theta3[2])

print('\n',X[2])
print('\n',Z[2])








 88.09111386083484

 1.9088861391651584

 48.14867445060685

 41.036569057366385

 20.0


In [403]:
byte1[2], byte2[2] = break_into_two(Servo1[2])
byte3[2], byte4[2] = break_into_two(Servo2[2])
byte5[2], byte6[2] = break_into_two(Servo3[2])
byte7[2], byte8[2] = break_into_two(Servo4[2])
byte9[2], byte10[2] = break_into_two(Servo5[2])

In [404]:
WriteByte(ser, int(byte1[2]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[2]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[2]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[2]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[2]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[2]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[2]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[2]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[2]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[2]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")

 servo1=My int: 1239  
 servo1=My int2: 1010  
 servo1=My int3: 1232  
 servo1=My int4: 887  
 servo1=My int5: 0  


In [405]:
#Step 3 - Grab Object
Theta1[3] = Theta1[2]
X[3] = X[2]
Z[3] = Height2
Theta2[3], Theta3[3] = XZLocation(L1, L2, X[3], Z[3])
Theta4[3] = -1 * (180 - GrabberAngle(L1, L2, X[3], Z[3]))
Servo1[3] = theta2interpolation(Theta1[3])
Servo2[3] = theta2interpolation(Theta2[3])
Servo3[3] = theta3interpolation(Theta3[3])
Servo4[3] = theta4interpolation(Theta4[3])

print('\n',Servo1[3])
print('\n',Servo2[3])
print('\n',Servo3[3])
print('\n',Servo4[3])
print('\n',Servo5[3])


 1239.028144383989

 1037.9130551579547

 1262.5363398776178

 854.5043338104545

 0.0


In [406]:
byte1[3], byte2[3] = break_into_two(Servo1[3])
byte3[3], byte4[3] = break_into_two(Servo2[3])
byte5[3], byte6[3] = break_into_two(Servo3[3])
byte7[3], byte8[3] = break_into_two(Servo4[3])
byte9[3], byte10[3] = break_into_two(Servo5[3])

In [407]:
WriteByte(ser, int(byte1[3]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[3]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[3]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[3]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[3]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[3]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[3]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[3]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[3]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[3]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")

 servo1=My int: 1239  
 servo1=My int2: 1037  
 servo1=My int3: 1262  
 servo1=My int4: 854  
 servo1=My int5: 0  


In [290]:
#Step 4 - Go to clearance distance to be able to move around obstacle
Theta1[4] = Theta1[3]
X[4] = (ObstacleY**2 + ObstacleX**2)**.5 - CL
Z[4] = Height1
if X[4] < (L1**2 + L2**2):
    ClearanceHeight = findz(L1, L2, X[4])
    Z[4] = ClearanceHeight
Theta2[4], Theta3[4] = XZLocation(L1, L2, X[4], Z[4])
if Theta3[4] == 90:
    Theta3[4] = -90
    Theta2[4] = Theta2[4] +90
Theta4[4] = -1 * (180 - GrabberAngle(L1, L2, X[4], Z[4]))

if Theta4[4] < -90:
    Theta4[4] = -90
Servo1[4] = theta2interpolation(Theta1[4])
Servo2[4] = theta2interpolation(Theta2[4])
Servo3[4] = theta3interpolation(Theta3[4])
Servo4[4] = theta4interpolation(Theta4[4])

print('\n',Servo1[4])
print('\n',Servo2[4])
print('\n',Servo3[4])
print('\n',Servo4[4])
print('\n',Servo5[4])


 1250.0

 1682.0806104051594

 2000.0

 1000.0

 0.0


In [291]:
byte1[4], byte2[4] = break_into_two(Servo1[4])
byte3[4], byte4[4] = break_into_two(Servo2[4])
byte5[4], byte6[4] = break_into_two(Servo3[4])
byte7[4], byte8[4] = break_into_two(Servo4[4])
byte9[4], byte10[4] = break_into_two(Servo5[4])

In [292]:
WriteByte(ser, int(byte1[4]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[4]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[4]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[4]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[4]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[4]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[4]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[4]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[4]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[4]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")

 servo1=My int: 1250  
 servo1=My int2: 1682  
 servo1=My int3: 2000  
 servo1=My int4: 1000  
 servo1=My int5: 0  


In [293]:
#Step 5 - Go to correct angle to drop off object
Theta1[5] =  rtd * np.arctan2(DropOffY, DropOffX)
X[5] = X[4]
Z[5] = Z[4]
Theta2[5] = Theta2[4]
Theta3[5] = Theta3[4]
Theta4[5] = Theta4[4]

Servo1[5] = theta2interpolation(Theta1[5])
Servo2[5] = theta2interpolation(Theta2[5])
Servo3[5] = theta3interpolation(Theta3[5])
Servo4[5] = theta4interpolation(Theta4[5])

print('\n',Servo1[5])
print('\n',Servo2[5])
print('\n',Servo3[5])
print('\n',Servo4[5])
print('\n',Servo5[5])


 1738.6366195061821

 1682.0806104051594

 2000.0

 1000.0

 0.0


In [294]:
byte1[5], byte2[5] = break_into_two(Servo1[5])
byte3[5], byte4[5] = break_into_two(Servo2[5])
byte5[5], byte6[5] = break_into_two(Servo3[5])
byte7[5], byte8[5] = break_into_two(Servo4[5])
byte9[5], byte10[5] = break_into_two(Servo5[5])

In [295]:
WriteByte(ser, int(byte1[5]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[5]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[5]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[5]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[5]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[5]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[5]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[5]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[5]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[5]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")

 servo1=My int: 1738  
 servo1=My int2: 1682  
 servo1=My int3: 2000  
 servo1=My int4: 1000  
 servo1=My int5: 0  


In [296]:
#Step 6 - Go to correct location to drop off object
Theta1[6] = Theta1[5]
X[6] = (DropOffY**2 + DropOffX**2)**.5
Z[6] = Height1
Theta2[6], Theta3[6] = XZLocation(L1, L2, X[6], Z[6])
Theta4[6] = -1 * (180 - GrabberAngle(L1, L2, X[6], Z[6]))

Servo1[6] = theta2interpolation(Theta1[6])
Servo2[6] = theta2interpolation(Theta2[6])
Servo3[6] = theta3interpolation(Theta3[6])
Servo4[6] = theta4interpolation(Theta4[6])

print('\n',Servo1[6])
print('\n',Servo2[6])
print('\n',Servo3[6])
print('\n',Servo4[6])
print('\n',Servo5[6])


 1738.6366195061821

 1165.539612673919

 1910.991019489717

 1165.5396126739192

 0.0


In [297]:
byte1[6], byte2[6] = break_into_two(Servo1[6])
byte3[6], byte4[6] = break_into_two(Servo2[6])
byte5[6], byte6[6] = break_into_two(Servo3[6])
byte7[6], byte8[6] = break_into_two(Servo4[6])
byte9[6], byte10[6] = break_into_two(Servo5[6])

In [298]:
WriteByte(ser, int(byte1[6]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[6]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[6]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[6]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[6]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[6]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[6]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[6]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[6]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[6]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")

 servo1=My int: 1738  
 servo1=My int2: 1165  
 servo1=My int3: 1910  
 servo1=My int4: 1165  
 servo1=My int5: 0  


In [299]:
#Step 7 - Drop Off object
Theta1[7] = Theta1[6]
X[7] = X[6]
Z[7] = Height2
Theta2[7], Theta3[7] = XZLocation(L1, L2, X[7], Z[7])
Theta4[7] = -1 * (180 - GrabberAngle(L1, L2, X[7], Z[7]))

Servo1[7] = theta2interpolation(Theta1[7])
Servo2[7] = theta2interpolation(Theta2[7])
Servo3[7] = theta3interpolation(Theta3[7])
Servo4[7] = theta4interpolation(Theta4[7])

print('\n',Servo1[7])
print('\n',Servo2[7])
print('\n',Servo3[7])
print('\n',Servo4[7])
print('\n',Servo5[7])



 1738.6366195061821

 1146.6472486317548

 1904.6137187168783

 1146.6472486317548

 0.0


In [300]:
byte1[7], byte2[7] = break_into_two(Servo1[7])
byte3[7], byte4[7] = break_into_two(Servo2[7])
byte5[7], byte6[7] = break_into_two(Servo3[7])
byte7[7], byte8[7] = break_into_two(Servo4[7])
byte9[7], byte10[7] = break_into_two(Servo5[7])

In [301]:
WriteByte(ser, int(byte1[7]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[7]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[7]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[7]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[7]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[7]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[7]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[7]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[7]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[7]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")

 servo1=My int: 1738  
 servo1=My int2: 1146  
 servo1=My int3: 1904  
 servo1=My int4: 1146  
 servo1=My int5: 0  


In [70]:
myint1 = 1700
byte1, byte2 = break_into_two(myint1)

myint2 = 1600
byte3, byte4 = break_into_two(myint2)

myint3 = 1500
byte5, byte6 = break_into_two(myint3)

myint4 = 1400
byte7, byte8 = break_into_two(myint4)

myint5 = 1300
byte9, byte10 = break_into_two(myint5)

In [71]:
WriteByte(ser, int(byte1))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")


 servo1=My int: 1700  
 servo1=My int2: 1600  
 servo1=My int3: 1500  
 servo1=My int4: 1400  
 servo1=My int5: 1300  


In [47]:
ser.close()

In [ ]:
byte1[0], byte2[0] = break_into_two(Servo1[0])
byte3[0], byte4[0] = break_into_two(Servo2[0])
byte5[0], byte6[0] = break_into_two(Servo3[0])
byte7[0], byte8[0] = break_into_two(Servo4[0])
byte9[0], byte10[0] = break_into_two(Servo5[0])

In [ ]:
WriteByte(ser, int(byte1[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2[0]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte3[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte5[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte6[0]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte7[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte8[0]))  # servo 1 MSB
time.sleep(0.05)

WriteByte(ser, int(byte9[0]))  # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte10[0]))  # servo 1 MSB
time.sleep(0.05)

line1 = read_one_line(ser)  # servo 1 int echo
line2 = read_one_line(ser)  # servo 1 int echo
line3 = read_one_line(ser)
line4 = read_one_line(ser)
line5 = read_one_line(ser)

print(f" servo1={line1}  ")

print(f" servo1={line2}  ")
print(f" servo1={line3}  ")
print(f" servo1={line4}  ")
print(f" servo1={line5}  ")